# 01 — Classificação binária de diabetes (Pima)

Tech Challenge Fase 1 — FIAP Pós Tech · IA para Devs.

Esqueleto do notebook principal. Cada etapa abaixo será desenvolvida nas próximas features. A célula de preparação torna o notebook executável em local, Docker e Google Colab. **Métrica principal: recall** (falso negativo é o erro mais grave).

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/iigormatos/tech-challenge-fase1.git"
EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    PROJECT_ROOT = Path("/content/tech-challenge-fase1")
    if not PROJECT_ROOT.exists():
        !git clone -q {REPO_URL} {PROJECT_ROOT}
    %cd {PROJECT_ROOT}
    !pip install -q missingno shap
else:
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.config import RANDOM_STATE, DIABETES_CSV, PNEUMONIA_DIR, FIGURES_DIR, MODEL_DIR

import pandas, numpy, sklearn, shap
print("pandas", pandas.__version__, "| numpy", numpy.__version__,
      "| scikit-learn", sklearn.__version__, "| shap", shap.__version__)

## Etapa 1 — Contexto do problema

### Problema de negócio

Um hospital universitário quer **acelerar a triagem** de pacientes e apoiar decisões clínicas. A
proposta é um sistema de **suporte ao diagnóstico** que estima o risco de diabetes a partir de
exames clínicos de rotina, priorizando quem deve ser investigado primeiro.

### Objetivo do modelo

Classificar cada paciente como **positivo** (provável diabetes) ou **negativo**, produzindo também
uma **probabilidade** e uma **explicação** que auxiliem a priorização — não um veredito automático.

### Base de dados

Dataset **Pima Indians Diabetes**: mulheres de origem Pima, com 21 anos ou mais. São **768
registros** e **9 colunas** (8 atributos clínicos + a variável alvo).

### Dicionário de variáveis

| Variável | Descrição |
|---|---|
| `Pregnancies` | Número de gestações |
| `Glucose` | Glicose plasmática (teste oral de tolerância) |
| `BloodPressure` | Pressão diastólica (mm Hg) |
| `SkinThickness` | Espessura da dobra cutânea do tríceps (mm) |
| `Insulin` | Insulina sérica de 2h (µU/ml) |
| `BMI` | Índice de massa corporal (kg/m²) |
| `DiabetesPedigreeFunction` | Histórico familiar (função de pedigree) |
| `Age` | Idade (anos) |
| `Outcome` | **Alvo**: 1 = diabetes, 0 = não diabetes |

### Variável alvo e custo dos erros

O alvo é `Outcome`. O custo dos erros é **assimétrico**: um **falso negativo** (paciente diabético
classificado como saudável) é o erro **mais grave**, porque a pessoa deixa de ser investigada. Um
falso positivo gera, no pior caso, um exame adicional. Por isso a métrica principal do projeto será
o **recall** da classe positiva (detalhado nas etapas de modelagem e avaliação).

### Aviso

Este sistema é uma ferramenta de **apoio à decisão**, não de decisão automática. A saída é uma
**triagem/priorização** com probabilidade e explicação; **o médico sempre tem a palavra final**.


## Etapa 2 — Exploração de dados (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import TARGET, COLUNAS_NUMERICAS

df = pd.read_csv(DIABETES_CSV)
print("shape:", df.shape)
df.info()
df.head()

**Inferência sobre os dados**

A base tem 768 registros e 9 colunas, sem nulos declarados (`Non-Null Count` = 768 em todas) e tipos numéricos (7 `int64`, 2 `float64`). Porém há vários **zeros** em variáveis fisiológicas (visíveis já no `head`, ex.: `Insulin`=0, `SkinThickness`=0) que são ausências disfarçadas — tratadas na Etapa 3.

In [ ]:
# Estatísticas descritivas; abaixo, destaque para os mínimos iguais a zero
desc = df.describe().T
display(desc)

zeros_por_coluna = (df[COLUNAS_NUMERICAS] == 0).sum()
print("Zeros por variável (0 é impossível em Glucose/BloodPressure/SkinThickness/Insulin/BMI):")
print(zeros_por_coluna)

**Inferência sobre os dados**

As descritivas confirmam **mínimos = 0 biologicamente impossíveis** em `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` e `BMI`. As contagens de zero são: `Insulin` 374, `SkinThickness` 227, `BloodPressure` 35, `BMI` 11 e `Glucose` 5 (em `Pregnancies` o zero é válido). Há cauda longa/outliers: `Insulin` vai até 846 (média ~80) e `SkinThickness` até 99. Esses zeros viram ausentes na Etapa 3.

In [ ]:
# Balanceamento da variável alvo
contagem = df[TARGET].value_counts().sort_index()
proporcao = df[TARGET].value_counts(normalize=True).sort_index().round(3)
display(pd.DataFrame({"contagem": contagem, "proporção": proporcao}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=TARGET, data=df, ax=ax)
ax.set_title("Balanceamento da variável alvo (Outcome)")
ax.set_xlabel("Outcome (0 = não diabético, 1 = diabético)")
ax.set_ylabel("contagem")
plt.savefig(FIGURES_DIR / "balanceamento_classes.png", dpi=150, bbox_inches="tight")
plt.show()

**Inferência sobre os dados**

A variável alvo é **desbalanceada**: 500 negativos (65,1%) e 268 positivos (34,9%). Isso reforça por que a **acurácia engana** e o **recall** da classe positiva será a métrica principal — prever sempre 'não diabético' já acertaria ~65%.

In [ ]:
# Distribuição de cada variável numérica, geral e por classe (histogramas)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), COLUNAS_NUMERICAS):
    sns.histplot(data=df, x=col, hue=TARGET, kde=True, ax=ax, legend=(col == COLUNAS_NUMERICAS[0]))
    ax.set_title(col)
fig.suptitle("Distribuição por variável (hue = Outcome)")
fig.tight_layout()
plt.show()

**Inferência sobre os dados**

As distribuições diferem entre as classes: `Glucose`, `BMI` e `Age` tendem a valores mais altos no grupo positivo (`Outcome`=1), coerente com o ranking de correlação (Etapa 4). Os picos em zero de `Insulin`/`SkinThickness` refletem os ausentes disfarçados. `TODO: revisar` detalhes visuais de cada subplot.

In [ ]:
# Boxplots por classe (identificação de outliers)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), COLUNAS_NUMERICAS):
    sns.boxplot(data=df, x=TARGET, y=col, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Outcome")
fig.suptitle("Boxplots por classe — outliers apenas sinalizados (não removidos)")
fig.tight_layout()
plt.savefig(FIGURES_DIR / "boxplots_por_classe.png", dpi=150, bbox_inches="tight")
plt.show()

**Inferência sobre os dados**

Os boxplots mostram outliers em várias variáveis (ex.: `Insulin` até 846, `SkinThickness` até 99, `BMI` até 67,1). Em dados clínicos, valores extremos podem ser reais; por isso são **apenas sinalizados** aqui, não removidos. `TODO: revisar` a leitura fina por variável.

In [ ]:
# Pairplot das 4 variáveis mais correlacionadas com o alvo
corr_alvo = (df[COLUNAS_NUMERICAS + [TARGET]].corr()[TARGET]
             .drop(TARGET).abs().sort_values(ascending=False))
top4 = corr_alvo.head(4).index.tolist()
print("4 variáveis mais correlacionadas com Outcome:", top4)
sns.pairplot(df[top4 + [TARGET]], hue=TARGET, corner=True)
plt.show()

**Inferência sobre os dados**

As 4 variáveis mais correlacionadas com o alvo são **`Glucose`, `BMI`, `Age` e `Pregnancies`**. No pairplot há sobreposição entre as classes (não são linearmente separáveis), mas `Glucose` oferece a melhor separação parcial — indício de ser a preditora mais forte.

## Etapa 3 — Limpeza de dados

In [ ]:
import missingno as msno
from src.preprocess import (marcar_zeros_como_ausentes, criar_faixas_clinicas,
                            COLUNAS_ZERO_INVALIDO)

# Transformações determinísticas (sem estatística aprendida)
df_marcado = marcar_zeros_como_ausentes(df)
df_faixas = criar_faixas_clinicas(df_marcado)
print("Zeros impossíveis convertidos em NaN nas colunas:", COLUNAS_ZERO_INVALIDO)

In [ ]:
# Ausentes por variável (após marcar zeros impossíveis)
ausentes = pd.DataFrame({
    "ausentes": df_marcado.isna().sum(),
    "%": (df_marcado.isna().mean() * 100).round(1),
}).sort_values("ausentes", ascending=False)
display(ausentes)

**Inferência sobre os dados**

Após marcar os zeros impossíveis, os ausentes concentram-se em **`Insulin` (374; 48,7%)** e **`SkinThickness` (227; 29,6%)**; `BloodPressure` (35; 4,6%), `BMI` (11; 1,4%) e `Glucose` (5; 0,7%) têm pouca ausência. Isso justifica **manter** `Insulin`/`SkinThickness` e testar os dois cenários na modelagem, em vez de descartá-las.

In [ ]:
# Visualização dos ausentes
msno.matrix(df_marcado)
plt.savefig(FIGURES_DIR / "matriz_ausentes.png", dpi=150, bbox_inches="tight")
plt.show()

**Inferência sobre os dados**

A matriz do `missingno` mostra a ausência concentrada em `Insulin` e `SkinThickness`, frequentemente nos mesmos registros. `TODO: revisar` se o padrão sugere ausência não aleatória (MNAR), o que influenciaria a estratégia de imputação.

In [ ]:
# Duplicatas
print("Linhas duplicadas:", int(df.duplicated().sum()))

**Inferência sobre os dados**

Não há linhas duplicadas (0), então não é preciso remover registros repetidos.

In [ ]:
# Distribuição das faixas clínicas por classe
print("FaixaEtaria × Outcome (proporção por faixa):")
display(pd.crosstab(df_faixas["FaixaEtaria"], df_faixas[TARGET], normalize="index").round(3))
print("CategoriaIMC × Outcome (proporção por categoria):")
display(pd.crosstab(df_faixas["CategoriaIMC"], df_faixas[TARGET], normalize="index").round(3))

**Inferência sobre os dados**

A proporção de positivos **cresce com a idade** (21-30: 21,6% → 31-40: 48,4% → 41-50: 56,6%), com leve queda em 50+ (46,9%). Também **cresce com o IMC** (`Normal` 6,9% → `Sobrepeso` 22,3% → `Obesidade` 46,4%); a categoria `Abaixo` tem 0% de positivos, mas com pouquíssimos casos. As faixas capturam relações não lineares úteis para a discussão.

### Decisões de limpeza (para a modelagem — feature 003)

1. **Zeros impossíveis → ausentes**: em `Glucose, BloodPressure, SkinThickness, Insulin, BMI`, o valor 0 foi convertido em `NaN` (`marcar_zeros_como_ausentes`). Transformação determinística, já aplicada nesta etapa.
2. **Imputação**: NÃO é feita aqui. Na modelagem, imputar por **mediana** dentro do `Pipeline`, com `fit` apenas no treino (evita vazamento — Princípio II).
3. **`Insulin` e `SkinThickness`** (maiores % de ausentes): **mantidas**. A modelagem deve **testar os dois cenários** — com e sem essas colunas (e/ou com imputação) — comparando pelo recall.
4. **Outliers**: apenas **sinalizados** nesta etapa (boxplots da Etapa 2). Valor extremo pode ser clínico real; eventual remoção é decisão da modelagem.
5. **Faixas clínicas** (`FaixaEtaria`, `CategoriaIMC`): criadas por cortes fixos e **convivem** com `Age`/`BMI`. A modelagem decide se as usa (encoding) ou descarta.

## Etapa 4 — Análise de correlação

In [ ]:
# Matriz de correlação das numéricas, após zeros → NaN (pandas ignora NaN par a par)
corr = df_marcado[COLUNAS_NUMERICAS + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr.round(2), annot=True, cmap="coolwarm", center=0, linewidths=.5, ax=ax)
ax.set_title("Matriz de correlação (numéricas, após zeros → NaN)")
plt.savefig(FIGURES_DIR / "correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

**Inferência sobre os dados**

Com os zeros já como ausentes, a correlação linear com `Outcome` é liderada por `Glucose`; as demais variáveis têm correlação fraca a moderada entre si. Não há correlações extremas (nada próximo de 1), o que reduz o risco de redundância severa.

In [ ]:
# Ranking de correlação com a variável alvo
ranking = corr[TARGET].drop(TARGET).sort_values(ascending=False)
display(ranking.round(3).to_frame("correlação com Outcome"))

**Inferência sobre os dados**

Ranking de correlação com `Outcome`: `Glucose` 0,495 (isolada no topo), depois `BMI` 0,314, `Insulin` 0,303, `SkinThickness` 0,259, `Age` 0,238, `Pregnancies` 0,222; `DiabetesPedigreeFunction` (0,174) e `BloodPressure` (0,171) são as mais fracas. Todas positivas — nenhuma variável é protetora nesta base.

In [ ]:
# Multicolinearidade: pares de features com |correlação| > 0,5
import itertools
corr_feat = df_marcado[COLUNAS_NUMERICAS].corr().abs()
pares = [(a, b, round(corr_feat.loc[a, b], 3))
         for a, b in itertools.combinations(COLUNAS_NUMERICAS, 2)
         if corr_feat.loc[a, b] > 0.5]
print("Pares com |correlação| > 0,5:")
for a, b, v in sorted(pares, key=lambda x: -x[2]):
    print(f"  {a} × {b}: {v}")
if not pares:
    print("  nenhum par acima de 0,5")

**Inferência sobre os dados**

Pares com `|corr| > 0,5`: **`SkinThickness × BMI` (0,648)**, **`Glucose × Insulin` (0,581)** e **`Pregnancies × Age` (0,544)** — associações plausíveis (adiposidade; glicose/insulina; gestações/idade). Modelos lineares (Regressão Logística) podem sofrer com a multicolinearidade; a Random Forest é mais robusta. **Correlação não implica causalidade.**

**Correlação não implica causalidade.** As correlações acima indicam associação estatística, não relação de causa e efeito. Variáveis mais correlacionadas com `Outcome` são candidatas mais promissoras para a modelagem, mas a decisão final pondera desempenho e interpretabilidade.

### Por que esta EDA não gera vazamento de dados

Todas as transformações desta etapa são **determinísticas** (zeros → ausente; faixas por cortes fixos) e **nenhuma estatística é aprendida** dos dados (sem média, mediana, escalonamento ou `fit`). A exploração usa a base inteira apenas para **entender** os dados. A separação treino/teste, a imputação e o escalonamento acontecem somente na modelagem, dentro de um `Pipeline` com `fit` apenas no treino (Princípio II).

## Etapa 5 — Separação treino / validação / teste

O conjunto de **teste (20%)** é **reservado** para a avaliação final e **não** é usado em nenhuma decisão desta etapa. Não criamos um conjunto de validação separado: a **validação cruzada** (5 dobras estratificadas) sobre o treino cumpre esse papel — mais robusta numa base pequena. `X` contém os **dados brutos**; a limpeza (zeros→ausentes e faixas) acontece **dentro** do Pipeline (Etapa 6).

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET])   # dados brutos; limpeza ocorre no Pipeline
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("\nProporção da classe positiva (Outcome=1):")
print("  geral :", round(float(y.mean()), 3))
print("  treino:", round(float(y_train.mean()), 3))
print("  teste :", round(float(y_test.mean()), 3))

**Inferência sobre os dados**

O split preservou a proporção da classe positiva: **0,349** no geral, **0,349** no treino e **0,351** no teste (diferença de 0,2 p.p.), confirmando a estratificação. Treino com **614** amostras, teste com **154** — este último **selado** para a feature 004.

## Etapa 6 — Pipeline de pré-processamento

Cada modelo entra no **mesmo pipeline** (`build_pipeline`): `limpeza` (determinística: zeros→ausentes e faixas) → `prep` (imputação por mediana + escalonamento das numéricas; imputação pela moda + one-hot das categóricas) → `clf`. Como o `fit` do imputador, do escalonador e do codificador roda **dentro** da validação cruzada, ele é ajustado **apenas no treino de cada dobra** — sem vazamento. As transformações determinísticas ficam no pipeline para que o **modelo final receba dados brutos** (argumento de uso clínico real). Os dois cenários (`completo` e `sem_insulin_skin`) diferem só por `Insulin`/`SkinThickness`; as faixas entram em ambos.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from src.preprocess import build_pipeline, CENARIOS


def novos_modelos():
    """Fábrica de estimadores novos (evita reusar objetos já ajustados)."""
    return {
        "LogReg": LogisticRegression(class_weight="balanced", max_iter=1000,
                                     random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(),
        "RandomForest": RandomForestClassifier(class_weight="balanced",
                                               random_state=RANDOM_STATE),
    }

# Exemplo: pipeline da Regressão Logística no cenário completo
c = CENARIOS["completo"]
exemplo = build_pipeline(novos_modelos()["LogReg"], c["num"], c["cat"])
exemplo

**Inferência sobre os dados**

O pipeline tem 3 passos (`limpeza` → `prep` → `clf`) e recebe **dados brutos**. Como o `fit` do pré-processamento roda **dentro** da validação cruzada, cada dobra ajusta imputação, escala e encoding apenas no seu treino — sem vazamento (Princípio II).

## Etapa 7 — Modelagem e tuning

### Baseline (sem tuning)

Os 3 modelos são avaliados nos 2 cenários sob a **mesma** validação cruzada (`StratifiedKFold` de 5 dobras, definido uma vez em `cv`), sempre com **`X_train`/`y_train`** — o teste permanece selado. Métricas com **média ± desvio**; o **recall** é a principal.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from src.evaluate import SCORING, resumir_cv, tabela_comparativa

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # mesmo objeto p/ tudo

resumos_baseline = []
for nome_cen, cols in CENARIOS.items():
    for nome_clf, clf in novos_modelos().items():
        pipe = build_pipeline(clf, cols['num'], cols['cat'])
        res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING)
        resumos_baseline.append(resumir_cv(res, nome_clf + ' - ' + nome_cen))

tabela_baseline = tabela_comparativa(resumos_baseline)
tabela_baseline

**Inferência sobre os dados**

No baseline, a **Regressão Logística** lidera recall/F2 nos dois cenários (recall ≈ 0,75; F2 ≈ 0,72) e tem o maior ROC-AUC (~0,85). **KNN** e **Random Forest** têm mais precisão, mas recall bem menor (~0,58–0,60), inadequado ao foco clínico (falso negativo é o pior erro). Os dois cenários são praticamente equivalentes.

### Ajuste de hiperparâmetros (tuning)

`GridSearchCV` com `refit="f2"` (F-beta, β=2) para os 3 modelos nos 2 cenários, na **mesma** `cv` e sempre com **`X_train`/`y_train`**. O **F2** orienta ao recall sem colapsar a precisão; o **recall** segue como métrica principal reportada. O ajuste ocorre **só no treino** (Princípio II).

In [ ]:
from sklearn.model_selection import GridSearchCV

grades = {
    'LogReg': {'clf__C': [0.01, 0.1, 1, 10]},
    'KNN': {'clf__n_neighbors': list(range(3, 32, 2)),
            'clf__weights': ['uniform', 'distance']},
    'RandomForest': {'clf__n_estimators': [100, 300],
                     'clf__max_depth': [None, 5, 10],
                     'clf__min_samples_leaf': [1, 3, 5]},
}

buscas = {}
resumos_ajustado = []
for nome_cen, cols in CENARIOS.items():
    for nome_clf, clf in novos_modelos().items():
        pipe = build_pipeline(clf, cols['num'], cols['cat'])
        gs = GridSearchCV(pipe, grades[nome_clf], scoring=SCORING, refit='f2',
                          cv=cv, n_jobs=-1)
        gs.fit(X_train, y_train)
        buscas[(nome_clf, nome_cen)] = gs
        # estimativa de CV do melhor config (reaproveita resumir_cv)
        res_best = cross_validate(gs.best_estimator_, X_train, y_train, cv=cv, scoring=SCORING)
        resumos_ajustado.append(resumir_cv(res_best, nome_clf + ' - ' + nome_cen))

tabela_ajustado = tabela_comparativa(resumos_ajustado)
tabela_ajustado

In [ ]:
import pandas as pd

tab_base = tabela_baseline.copy(); tab_base.insert(0, 'fase', 'baseline')
tab_aj = tabela_ajustado.copy(); tab_aj.insert(0, 'fase', 'ajustado')
tabela_final = pd.concat([tab_base, tab_aj]).sort_index()

CSV_COMPARATIVO = FIGURES_DIR.parent / 'comparativo_modelos.csv'
tabela_final.to_csv(CSV_COMPARATIVO)
print('Tabela final salva em:', CSV_COMPARATIVO)
tabela_final

**Inferência sobre os dados**

Após o tuning os modelos se aproximam: **Random Forest** e **Regressão Logística** empatam tecnicamente no topo (F2 ≈ 0,73; recall ≈ 0,75–0,77), **dentro de 1 desvio padrão**. O **KNN** segue atrás (F2 ≈ 0,60–0,63). O cenário **`sem_insulin_skin`** iguala ou supera o `completo`, confirmando que descartar `Insulin`/`SkinThickness` (muitos ausentes) não prejudica e ainda simplifica o modelo.

### KNN — desempenho por número de vizinhos

In [ ]:
# Desempenho do KNN por numero de vizinhos (cenario completo)
gs_knn = buscas[('KNN', 'completo')]
res_knn = pd.DataFrame(gs_knn.cv_results_)
w = gs_knn.best_params_['clf__weights']
sub = res_knn[res_knn['param_clf__weights'] == w].sort_values('param_clf__n_neighbors')
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sub['param_clf__n_neighbors'], sub['mean_test_recall'], marker='o', label='recall')
ax.plot(sub['param_clf__n_neighbors'], sub['mean_test_f2'], marker='s', label='F2')
ax.set_xlabel('n_neighbors'); ax.set_ylabel('media da CV')
ax.set_title(f'KNN (weights={w}) - desempenho por numero de vizinhos')
ax.legend(); ax.grid(alpha=.3)
plt.savefig(FIGURES_DIR / 'knn_vizinhos.png', dpi=150, bbox_inches='tight')
plt.show()

**Inferência sobre os dados**

O KNN atinge o melhor F2 com `weights=distance` e **`n_neighbors=7`** (F2 ≈ 0,63). Para `k` muito pequeno há sobreajuste (recall/F2 instáveis) e, para `k` grande, o desempenho decai. Mesmo no melhor ponto, o KNN fica **abaixo** de Regressão Logística e Random Forest.

### Escolha do modelo final e do cenário

**Escolhido: Regressão Logística — cenário `sem_insulin_skin`** (`C=10`).

Justificativa (números da **validação cruzada no treino**):

- **Empate técnico no topo**: Random Forest `sem_insulin_skin` (F2 0,737 ± 0,017; recall 0,766 ± 0,017) e Regressão Logística `sem_insulin_skin` (F2 0,732 ± 0,030; recall 0,757 ± 0,052) diferem **menos de 1 desvio padrão** em F2 e recall — estatisticamente equivalentes.
- **Critério de desempate = interpretabilidade** (FR-012 / Princípio IV): a Regressão Logística é linear e fornece **coeficientes** diretamente interpretáveis, vantajoso num sistema de apoio ao diagnóstico. Ainda apresenta o **maior ROC-AUC** (0,850).
- **Cenário `sem_insulin_skin`**: iguala/supera o `completo` na CV e é **mais simples** — descarta as duas variáveis de muitos ausentes (`Insulin` ~49%, `SkinThickness` ~30%), reduzindo a dependência de exames frequentemente faltantes na triagem.
- **Métricas do modelo escolhido (CV)**: **recall 0,757** (métrica principal), F2 0,732 (critério de tuning), ROC-AUC 0,850.

O modelo final é ajustado no treino e salvo em `models/modelo_final.joblib` para a avaliação no teste (feature 004).

In [ ]:
import joblib

# Escolha do modelo final: empate tecnico (< 1 desvio) no topo entre RandomForest e LogReg
# no cenario sem_insulin_skin -> pelo FR-012 escolhemos o MAIS INTERPRETAVEL (LogReg).
MODELO_FINAL_CHAVE = ('LogReg', 'sem_insulin_skin')
gs_final = buscas[MODELO_FINAL_CHAVE]
modelo_final = gs_final.best_estimator_
print('Modelo/cenario final:', MODELO_FINAL_CHAVE)
print('Melhores hiperparametros:', gs_final.best_params_)
print('F2 medio (CV):', round(gs_final.best_score_, 3))
joblib.dump(modelo_final, MODEL_DIR / 'modelo_final.joblib')
print('Pipeline final salvo em:', MODEL_DIR / 'modelo_final.joblib')

**Inferência sobre os dados**

O `modelo_final` salvo é a **Regressão Logística** no cenário **`sem_insulin_skin`** (`C=10`), conforme a escolha justificada acima. O pipeline aceita **dados brutos** e está pronto para a avaliação no teste (feature 004), ainda intocado.

## Etapa 8 — Avaliação

## Etapa 9 — Interpretação (Feature Importance + SHAP)

## Etapa 10 — Discussão crítica

## Etapa 11 — Complemento opcional: K-means / PCA